In [ ]:
"""
stage0_events.py  --  Stage 0: event extraction for the MNQ JMA pivot intensity model.
Input: single SC-export parquet, MNQ and TICK columns on the same bar grid.

RULES (traceability)
  S0.1  Sign of a series is sign(diff), with zero-runs carried forward from the last
        non-zero sign. Leading run before the first non-zero sign is 0 (masked).
  S0.2  A pivot (extremum) of series x at bar k is emitted as an EVENT at bar k+1,
        the first bar at which the turn is knowable. event_bar is the only time index
        used downstream. extremum_bar is retained for diagnostics only.
  S0.3  The target (MNQ JMA pivot) obeys S0.2 identically: target at bar t <=>
        jma_leg_dir[t] != jma_leg_dir[t-1]. Hence the target is causal at t.
  S0.4  Stage 2 features at bar t must use exciting events with event_bar <= t-1.
        This file does not enforce that; it only guarantees event_bar is knowable-at.
  S0.5  Polarity: +1 = local maximum at extremum_bar, -1 = local minimum.
  S0.6  jma_leg_dir at an event is read at event_bar (causal). opposing = polarity *
        jma_leg_dir; +1 = opposing the leg, -1 = confirming, 0 = leg dir undefined.
  S0.7  No cross-session spillage: sign, pivots, legs and leg_age are computed per
        session_date. First WARMUP_BARS bars of each session are masked from both
        event emission and target counting.
  S0.8  All columns live on one bar grid in one file, each computed by SC at that
        bar's close => co-knowable at bar close. No join, no cross-column lag.
  S0.9  Diagnostics report the recall ceiling: fraction of target events preceded by
        an opposing pivot of stream s within lag support L. This upper-bounds the
        recall of ANY causal model using stream s with support L.
  S0.10 d1 centerline for the sign-agreement diagnostic is auto-detected: 100 if
        d1 is strictly positive with median in (90, 110) (SC ratio momentum),
        else 0 (point momentum). Affects the diagnostic only; pivots and all
        downstream features are centerline-free.
"""

import argparse
import json
import numpy as np
import pandas as pd

# ---------------------------------------------------------------- config

COLS = {
    "timestamp": "timestamp",
    "jma": "JMA",
    "d1": "jmaD1",
    "d2": "jmaD2",
    "t_d1": "tickJmaD1",
    "t_d2": "tickJmaD2",
}

TZ = "America/New_York"
WARMUP_BARS = 30
LAG_SUPPORTS = [1, 2, 3, 5, 8, 13, 21, 34]

STREAMS = [("MNQ_D1", "d1"), ("MNQ_D2", "d2"), ("TICK_D1", "t_d1"), ("TICK_D2", "t_d2")]


# ---------------------------------------------------------------- primitives

def signfill(x):
    """S0.1"""
    d = np.zeros(len(x), dtype=np.float64)
    d[1:] = np.diff(x)
    s = np.sign(d).astype(np.int8)
    pos = np.maximum.accumulate(np.where(s != 0, np.arange(len(s)), -1))
    out = np.zeros(len(s), dtype=np.int8)
    ok = pos >= 0
    out[ok] = s[pos[ok]]
    return out


def pivots(s):
    """S0.2, S0.5. s is a signfill array. Returns (event_bar, extremum_bar, polarity)."""
    if len(s) < 2:
        e = np.empty(0, dtype=np.int64)
        return e, e, np.empty(0, dtype=np.int8)
    turn = (s[1:] != s[:-1]) & (s[1:] != 0) & (s[:-1] != 0)
    event_bar = np.nonzero(turn)[0] + 1
    return event_bar, event_bar - 1, s[event_bar - 1]


# ---------------------------------------------------------------- load

def load(data_path, input_tz):
    df = pd.read_parquet(data_path)
    df = df.rename(columns={v: k for k, v in COLS.items()})
    df = df[list(COLS.keys())].copy()
    ts = pd.to_datetime(df["timestamp"])
    if ts.dt.tz is None:                                            # S0.8 assumption
        ts = ts.dt.tz_localize(input_tz)
    df["timestamp"] = ts.dt.tz_convert("UTC")
    df = df.sort_values("timestamp").reset_index(drop=True)
    df["session_date"] = df["timestamp"].dt.tz_convert(TZ).dt.date
    n0 = len(df)
    df = df.dropna(subset=["jma", "d1", "d2", "t_d1", "t_d2"]).reset_index(drop=True)
    df["bar_index"] = np.arange(len(df), dtype=np.int64)
    return df, n0 - len(df)


# ---------------------------------------------------------------- extraction

def extract(df):
    ev_rows = []
    bar_frames = []

    for sess, g in df.groupby("session_date", sort=True):
        g = g.reset_index(drop=True)
        if len(g) <= WARMUP_BARS + 2:
            continue
        gbar = g["bar_index"].to_numpy()

        jma = g["jma"].to_numpy(np.float64)
        leg_dir = signfill(jma)                                     # S0.1
        t_ev, t_ex, t_pol = pivots(signfill(jma))                   # S0.3 (target)

        keep = t_ev >= WARMUP_BARS                                  # S0.7
        t_ev, t_ex, t_pol = t_ev[keep], t_ex[keep], t_pol[keep]

        is_target = np.zeros(len(g), dtype=bool)
        is_target[t_ev] = True
        leg_id = np.cumsum(is_target)
        starts = np.concatenate(([0], t_ev))
        leg_start = starts[leg_id]
        leg_age = np.arange(len(g)) - leg_start
        leg_amp = np.abs(jma - jma[leg_start])

        bar_frames.append(pd.DataFrame({
            "bar_index": gbar,
            "timestamp": g["timestamp"].to_numpy(),
            "session_date": sess,
            "jma": jma,
            "d1": g["d1"].to_numpy(np.float64),
            "jma_leg_dir": leg_dir,
            "leg_id": leg_id.astype(np.int32),
            "leg_age": leg_age.astype(np.int32),
            "leg_amp": leg_amp,
            "is_target": is_target,
            "warm": np.arange(len(g)) < WARMUP_BARS,
        }))

        for name, col in STREAMS + [("MNQ_JMA_SELF", "jma")]:
            x = g[col].to_numpy(np.float64)
            e, k, pol = pivots(signfill(x))
            m = e >= WARMUP_BARS                                    # S0.7
            e, k, pol = e[m], k[m], pol[m]
            if len(e) == 0:
                continue
            xk = x[k]
            prev_e = np.concatenate(([-1], e[:-1]))
            prev_xk = np.concatenate(([np.nan], xk[:-1]))
            ev_rows.append(pd.DataFrame({
                "session_date": sess,
                "stream": name,
                "event_bar": gbar[e],
                "extremum_bar": gbar[k],
                "timestamp": g["timestamp"].to_numpy()[e],
                "polarity": pol,                                    # S0.5
                "jma_leg_dir": leg_dir[e],                          # S0.6
                "opposing": (pol * leg_dir[e]).astype(np.int8),     # S0.6
                "value": xk,
                "prev_leg_amp": np.abs(xk - prev_xk),
                "bars_since_prev": np.where(prev_e < 0, -1, e - prev_e).astype(np.int32),
                "leg_age_at_event": leg_age[e].astype(np.int32),
            }))

    bars = pd.concat(bar_frames, ignore_index=True)
    events = pd.concat(ev_rows, ignore_index=True).sort_values(["event_bar", "stream"]).reset_index(drop=True)
    events["stream_id"] = events["stream"].astype("category").cat.codes.astype(np.int8)
    return bars, events


# ---------------------------------------------------------------- diagnostics

def diagnostics(bars, events, dropped):
    tgt = np.sort(bars.loc[bars["is_target"], "bar_index"].to_numpy())
    n_bars = int((~bars["warm"]).sum())

    d1 = bars["d1"].to_numpy()
    center = 100.0 if (np.nanmin(d1) > 0 and 90.0 < np.nanmedian(d1) < 110.0) else 0.0   # S0.10
    agree = float((np.sign(d1 - center) == bars["jma_leg_dir"].to_numpy()).mean())

    d = {
        "n_bars": int(len(bars)),
        "n_bars_ex_warmup": n_bars,
        "n_sessions": int(bars["session_date"].nunique()),
        "rows_dropped_nan": int(dropped),
        "n_target": int(len(tgt)),
        "target_rate_per_1k": round(1000.0 * len(tgt) / max(n_bars, 1), 3),
        "median_leg_bars": float(np.median(np.diff(tgt))) if len(tgt) > 2 else None,
        "d1_center_detected": center,
        "d1_sign_agrees_leg_dir": agree,
        "streams": {},
        "recall_ceiling": {},
        "precision_base": {},
    }

    for name, g in events.groupby("stream"):
        eb = np.sort(g["event_bar"].to_numpy())
        d["streams"][name] = {
            "n": int(len(g)),
            "rate_per_1k": round(1000.0 * len(g) / max(n_bars, 1), 3),
            "ratio_to_target": round(len(g) / max(len(tgt), 1), 2),
            "median_inter_event_bars": float(np.median(np.diff(eb))) if len(eb) > 2 else None,
            "frac_opposing": float((g["opposing"] == 1).mean()),
        }

    opp = events[events["opposing"] == 1]
    for name, g in opp.groupby("stream"):                           # S0.9
        eb = np.sort(g["event_bar"].to_numpy())
        rec, prec = {}, {}
        for L in LAG_SUPPORTS:
            lo = np.searchsorted(eb, tgt - L, side="left")
            hi = np.searchsorted(eb, tgt, side="left")
            rec[L] = round(float((hi > lo).mean()), 4)
            a = np.searchsorted(tgt, eb + 1, side="left")
            b = np.searchsorted(tgt, eb + L, side="right")
            prec[L] = round(float((b > a).mean()), 4)
        d["recall_ceiling"][name] = rec
        d["precision_base"][name] = prec

    eb = np.sort(opp["event_bar"].unique())
    d["recall_ceiling"]["ANY_OPPOSING"] = {
        L: round(float((np.searchsorted(eb, tgt, "left") > np.searchsorted(eb, tgt - L, "left")).mean()), 4)
        for L in LAG_SUPPORTS
    }
    return d


# ---------------------------------------------------------------- main

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", required=True)
    p.add_argument("--frame", type=int, required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--input-tz", default=TZ)
    a = p.parse_args()

    df, dropped = load(a.data, a.input_tz)
    bars, events = extract(df)
    diag = diagnostics(bars, events, dropped)

    bars.to_parquet(f"{a.out}/bars_{a.frame}s.parquet", index=False)
    events.to_parquet(f"{a.out}/events_{a.frame}s.parquet", index=False)
    with open(f"{a.out}/stage0_diagnostics_{a.frame}s.json", "w") as f:
        json.dump(diag, f, indent=2, default=str)
    print(json.dumps(diag, indent=2, default=str))


if __name__ == "__main__":
    main()